# Módulo 2 · Clase 5 — Proyecto: versionar el sistema de biblioteca

**Curso de Inteligencia Artificial — Nivel básico**

---

## Objetivo del proyecto

En este proyecto integrador aplicamos **todo lo aprendido en el Módulo 2** al sistema de biblioteca que desarrollamos en la Clase 12 del Módulo 1.

Vamos a:

1. Inicializar un repositorio Git para el sistema de biblioteca.
2. Construir su historial de forma profesional (commits convencionales, mensajes claros).
3. Usar ramas para cada funcionalidad.
4. Etiquetar versiones con `git tag`.
5. Publicar en GitHub con la estructura correcta.
6. Generar informes del historial con Python.

```
ESTRUCTURA DEL PROYECTO:

sistema_biblioteca/
├── excepciones.py      # Excepciones del dominio
├── material.py         # ABC Material + Libro, Revista, DVD
├── usuario.py          # Clase Usuario
├── prestamo.py         # Clase Prestamo
├── biblioteca.py       # Clase Biblioteca (núcleo)
├── persistencia.py     # Guardado/carga JSON
├── cli.py              # Menú interactivo
├── tests/
│   └── test_biblioteca.py
├── .gitignore
├── README.md
└── requirements.txt
```

> 💡 Este proyecto simula un historial realista de 4 semanas de desarrollo, donde cada commit cuenta una historia clara.

---
## Fase 1 — Inicializar el repositorio

Empezamos desde cero: un directorio vacío que se convierte en un proyecto Git profesional.

In [ ]:
import os, shutil
from pathlib import Path

BLIB = Path.home() / "sistema_biblioteca"

if BLIB.exists():
    shutil.rmtree(BLIB)

BLIB.mkdir()
(BLIB / "tests").mkdir()
os.chdir(BLIB)

# Inicializar Git
!git init -b main
!git config user.name "Alumno Curso"
!git config user.email "alumno@curso.io"

print(f"Directorio: {Path.cwd()}")

In [ ]:
# .gitignore profesional para Python
Path(".gitignore").write_text("""\
# Python
__pycache__/
*.py[cod]
*.pyo
*.pyd
.Python

# Entornos virtuales
venv/
env/
.venv/

# Datos (no versionar archivos de datos generados)
datos/
*.json
!package.json

# IDE
.vscode/
.idea/
*.swp

# macOS
.DS_Store

# Tests
.pytest_cache/
htmlcov/
.coverage
""", encoding="utf-8")

!git add .gitignore
!git commit -m "chore: añadir .gitignore para Python"
print("Primer commit creado")

In [ ]:
Path("README.md").write_text("""\
# Sistema de Gestión de Biblioteca

Sistema para gestionar el catálogo, usuarios y préstamos de una biblioteca.
Proyecto integrador del Módulo 1 del Curso de IA.

## Instalación

```bash
git clone https://github.com/TU-USUARIO/sistema-biblioteca.git
cd sistema-biblioteca
pip install -r requirements.txt
```

## Uso

```bash
python cli.py
```

## Arquitectura

```
Material (ABC)
├── Libro
├── Revista
└── DVD

Biblioteca
├── catalogo: list[Material]
├── usuarios: list[Usuario]
└── prestamos: list[Prestamo]
```

## Licencia

MIT
""", encoding="utf-8")

Path("requirements.txt").write_text("# Sin dependencias externas\n", encoding="utf-8")

!git add README.md requirements.txt
!git commit -m "docs: añadir README con arquitectura y requirements"

print("\nHistorial inicial:")
!git log --oneline

---
## Fase 2 — Desarrollar el núcleo en ramas

Cada módulo del sistema se desarrolla en su propia rama, como lo haría un equipo real.

In [ ]:
# feature/excepciones — el sistema de excepciones del dominio
!git switch -c feature/excepciones

Path("excepciones.py").write_text("""\
class BibliotecaError(Exception):
    pass

class MaterialNoEncontrado(BibliotecaError):
    def __init__(self, isbn):
        super().__init__(f"Material con ISBN '{isbn}' no encontrado")
        self.isbn = isbn

class UsuarioNoEncontrado(BibliotecaError):
    def __init__(self, id_usuario):
        super().__init__(f"Usuario con ID '{id_usuario}' no encontrado")
        self.id_usuario = id_usuario

class MaterialNoDisponible(BibliotecaError):
    def __init__(self, isbn):
        super().__init__(f"Material '{isbn}' no disponible para préstamo")
        self.isbn = isbn

class LimitePrestamosAlcanzado(BibliotecaError):
    def __init__(self, id_usuario, limite):
        super().__init__(f"Usuario '{id_usuario}' ha alcanzado el límite de {limite} préstamos")
        self.id_usuario = id_usuario
        self.limite = limite

class PrestamoNoEncontrado(BibliotecaError):
    def __init__(self, id_prestamo):
        super().__init__(f"Préstamo '{id_prestamo}' no encontrado")
        self.id_prestamo = id_prestamo
""", encoding="utf-8")

!git add excepciones.py
!git commit -m "feat: jerarquía de excepciones del dominio biblioteca"

print("Rama feature/excepciones lista")

In [ ]:
# Integrar en main
!git switch main
!git merge feature/excepciones --no-ff -m "Merge: jerarquía de excepciones"
!git branch -d feature/excepciones

print("Historial:")
!git log --oneline --graph

In [ ]:
# feature/materiales — jerarquía de materiales con ABC
!git switch -c feature/materiales

Path("material.py").write_text("""\
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class Material(ABC):
    isbn: str
    titulo: str
    anio: int
    disponible: bool = field(default=True, init=False)

    @property
    @abstractmethod
    def tipo(self) -> str:
        pass

    @abstractmethod
    def descripcion_corta(self) -> str:
        pass

    def to_dict(self) -> dict:
        return {
            "tipo": self.tipo,
            "isbn": self.isbn,
            "titulo": self.titulo,
            "anio": self.anio,
            "disponible": self.disponible,
        }

    def __str__(self):
        estado = "disponible" if self.disponible else "prestado"
        return f"[{self.tipo}] {self.titulo} ({self.anio}) — {estado}"


@dataclass
class Libro(Material):
    autor: str = ""
    paginas: int = 0

    @property
    def tipo(self) -> str:
        return "Libro"

    def descripcion_corta(self) -> str:
        return f"{self.titulo} — {self.autor} ({self.paginas} págs.)"

    def to_dict(self) -> dict:
        d = super().to_dict()
        d.update({"autor": self.autor, "paginas": self.paginas})
        return d


@dataclass
class Revista(Material):
    numero: int = 0
    editorial: str = ""

    @property
    def tipo(self) -> str:
        return "Revista"

    def descripcion_corta(self) -> str:
        return f"{self.titulo} #{self.numero} — {self.editorial}"

    def to_dict(self) -> dict:
        d = super().to_dict()
        d.update({"numero": self.numero, "editorial": self.editorial})
        return d


@dataclass
class DVD(Material):
    director: str = ""
    duracion_min: int = 0

    @property
    def tipo(self) -> str:
        return "DVD"

    def descripcion_corta(self) -> str:
        return f"{self.titulo} — dir. {self.director} ({self.duracion_min} min)"

    def to_dict(self) -> dict:
        d = super().to_dict()
        d.update({"director": self.director, "duracion_min": self.duracion_min})
        return d


TIPOS_MATERIAL = {
    "Libro": Libro,
    "Revista": Revista,
    "DVD": DVD,
}

def material_from_dict(d: dict) -> Material:
    tipo = d.pop("tipo")
    disponible = d.pop("disponible", True)
    clase = TIPOS_MATERIAL[tipo]
    m = clase(**d)
    m.disponible = disponible
    return m
""", encoding="utf-8")

!git add material.py
!git commit -m "feat: jerarquía Material (ABC) → Libro, Revista, DVD"

# Integrar en main
!git switch main
!git merge feature/materiales --no-ff -m "Merge: jerarquía de materiales"
!git branch -d feature/materiales

print("feature/materiales integrada")

In [ ]:
# feature/usuarios-y-prestamos
!git switch -c feature/usuarios-y-prestamos

Path("usuario.py").write_text("""\
from dataclasses import dataclass, field

@dataclass
class Usuario:
    id_usuario: str
    nombre: str
    email: str
    prestamos_activos: list = field(default_factory=list)

    def to_dict(self) -> dict:
        return {
            "id_usuario": self.id_usuario,
            "nombre": self.nombre,
            "email": self.email,
        }

    @classmethod
    def from_dict(cls, d: dict):
        return cls(
            id_usuario=d["id_usuario"],
            nombre=d["nombre"],
            email=d["email"],
        )

    def __str__(self):
        return f"Usuario({self.id_usuario}) {self.nombre} <{self.email}>"
""", encoding="utf-8")

!git add usuario.py
!git commit -m "feat: clase Usuario con serialización dict"

Path("prestamo.py").write_text("""\
from dataclasses import dataclass, field
from datetime import date

@dataclass
class Prestamo:
    id_prestamo: str
    id_usuario: str
    isbn_material: str
    fecha_prestamo: date = field(default_factory=date.today)
    fecha_devolucion: date | None = None
    activo: bool = True

    def devolver(self):
        self.fecha_devolucion = date.today()
        self.activo = False

    @property
    def dias_prestado(self) -> int:
        fin = self.fecha_devolucion or date.today()
        return (fin - self.fecha_prestamo).days

    def to_dict(self) -> dict:
        return {
            "id_prestamo": self.id_prestamo,
            "id_usuario": self.id_usuario,
            "isbn_material": self.isbn_material,
            "fecha_prestamo": self.fecha_prestamo.isoformat(),
            "fecha_devolucion": self.fecha_devolucion.isoformat() if self.fecha_devolucion else None,
            "activo": self.activo,
        }

    @classmethod
    def from_dict(cls, d: dict):
        p = cls(
            id_prestamo=d["id_prestamo"],
            id_usuario=d["id_usuario"],
            isbn_material=d["isbn_material"],
            fecha_prestamo=date.fromisoformat(d["fecha_prestamo"]),
            activo=d["activo"],
        )
        if d["fecha_devolucion"]:
            p.fecha_devolucion = date.fromisoformat(d["fecha_devolucion"])
        return p
""", encoding="utf-8")

!git add prestamo.py
!git commit -m "feat: clase Prestamo con gestión de fechas"

!git switch main
!git merge feature/usuarios-y-prestamos --no-ff -m "Merge: módulos Usuario y Prestamo"
!git branch -d feature/usuarios-y-prestamos
print("feature/usuarios-y-prestamos integrada")

In [ ]:
# feature/biblioteca — el núcleo de la aplicación
!git switch -c feature/biblioteca

Path("biblioteca.py").write_text("""\
import uuid
from material import Material, TIPOS_MATERIAL
from usuario import Usuario
from prestamo import Prestamo
from excepciones import (
    MaterialNoEncontrado, UsuarioNoEncontrado,
    MaterialNoDisponible, LimitePrestamosAlcanzado,
    PrestamoNoEncontrado,
)

LIMITE_PRESTAMOS = 3

class Biblioteca:
    def __init__(self, nombre: str):
        self.nombre = nombre
        self._catalogo: list[Material] = []
        self._usuarios: list[Usuario] = []
        self._prestamos: list[Prestamo] = []

    # ── Catálogo ──────────────────────────────────────────────
    def agregar_material(self, material: Material) -> None:
        self._catalogo.append(material)

    def buscar_material(self, isbn: str) -> Material:
        for m in self._catalogo:
            if m.isbn == isbn:
                return m
        raise MaterialNoEncontrado(isbn)

    def buscar_por_titulo(self, termino: str) -> list[Material]:
        t = termino.lower()
        return [m for m in self._catalogo if t in m.titulo.lower()]

    def listar_catalogo(self) -> list[Material]:
        return list(self._catalogo)

    # ── Usuarios ──────────────────────────────────────────────
    def registrar_usuario(self, usuario: Usuario) -> None:
        self._usuarios.append(usuario)

    def buscar_usuario(self, id_usuario: str) -> Usuario:
        for u in self._usuarios:
            if u.id_usuario == id_usuario:
                return u
        raise UsuarioNoEncontrado(id_usuario)

    # ── Préstamos ─────────────────────────────────────────────
    def prestar(self, id_usuario: str, isbn: str) -> Prestamo:
        usuario = self.buscar_usuario(id_usuario)
        material = self.buscar_material(isbn)

        if not material.disponible:
            raise MaterialNoDisponible(isbn)

        prestamos_activos = sum(1 for p in self._prestamos
                                if p.id_usuario == id_usuario and p.activo)
        if prestamos_activos >= LIMITE_PRESTAMOS:
            raise LimitePrestamosAlcanzado(id_usuario, LIMITE_PRESTAMOS)

        material.disponible = False
        prestamo = Prestamo(
            id_prestamo=str(uuid.uuid4())[:8],
            id_usuario=id_usuario,
            isbn_material=isbn,
        )
        self._prestamos.append(prestamo)
        return prestamo

    def devolver(self, id_prestamo: str) -> Prestamo:
        for p in self._prestamos:
            if p.id_prestamo == id_prestamo and p.activo:
                p.devolver()
                material = self.buscar_material(p.isbn_material)
                material.disponible = True
                return p
        raise PrestamoNoEncontrado(id_prestamo)

    def prestamos_activos(self) -> list[Prestamo]:
        return [p for p in self._prestamos if p.activo]

    def reporte_estado(self) -> dict:
        disponibles = sum(1 for m in self._catalogo if m.disponible)
        prestados = len(self._catalogo) - disponibles
        return {
            "total_materiales": len(self._catalogo),
            "disponibles": disponibles,
            "prestados": prestados,
            "total_usuarios": len(self._usuarios),
            "prestamos_activos": len(self.prestamos_activos()),
            "prestamos_totales": len(self._prestamos),
        }
""", encoding="utf-8")

!git add biblioteca.py
!git commit -m "feat: clase Biblioteca con catálogo, usuarios y préstamos"

!git switch main
!git merge feature/biblioteca --no-ff -m "Merge: núcleo Biblioteca"
!git branch -d feature/biblioteca

print("\nHistorial tras la Fase 2:")
!git log --oneline --graph

---
## Fase 3 — Persistencia, CLI y primera versión estable

In [ ]:
!git switch -c feature/persistencia

Path("persistencia.py").write_text("""\
import json
from pathlib import Path
from biblioteca import Biblioteca
from material import material_from_dict
from usuario import Usuario
from prestamo import Prestamo

def guardar(biblioteca: Biblioteca, ruta: str = "datos.json") -> None:
    datos = {
        "nombre": biblioteca.nombre,
        "catalogo": [m.to_dict() for m in biblioteca.listar_catalogo()],
        "usuarios": [u.to_dict() for u in biblioteca._usuarios],
        "prestamos": [p.to_dict() for p in biblioteca._prestamos],
    }
    Path(ruta).write_text(json.dumps(datos, ensure_ascii=False, indent=2),
                           encoding="utf-8")

def cargar(ruta: str = "datos.json") -> Biblioteca:
    texto = Path(ruta).read_text(encoding="utf-8")
    datos = json.loads(texto)
    bib = Biblioteca(datos["nombre"])
    for d in datos["catalogo"]:
        bib.agregar_material(material_from_dict(d.copy()))
    for d in datos["usuarios"]:
        bib.registrar_usuario(Usuario.from_dict(d))
    for d in datos["prestamos"]:
        bib._prestamos.append(Prestamo.from_dict(d))
    return bib
""", encoding="utf-8")

!git add persistencia.py
!git commit -m "feat: persistencia JSON — guardar y cargar biblioteca"

!git switch main
!git merge feature/persistencia --no-ff -m "Merge: persistencia JSON"
!git branch -d feature/persistencia
print("feature/persistencia integrada")

In [ ]:
!git switch -c feature/cli

Path("cli.py").write_text("""\
from biblioteca import Biblioteca
from material import Libro, Revista, DVD
from usuario import Usuario
from excepciones import BibliotecaError
import persistencia

ARCHIVO = "datos.json"

MENU = """
╔══════════════════════════════╗
║  SISTEMA DE BIBLIOTECA       ║
╠══════════════════════════════╣
║  1. Ver catálogo             ║
║  2. Añadir libro             ║
║  3. Registrar usuario        ║
║  4. Prestar material         ║
║  5. Devolver material        ║
║  6. Ver préstamos activos    ║
║  7. Ver reporte              ║
║  0. Salir (guardar)          ║
╚══════════════════════════════╝
"""

def main():
    from pathlib import Path
    if Path(ARCHIVO).exists():
        bib = persistencia.cargar(ARCHIVO)
        print(f"Biblioteca '{bib.nombre}' cargada.")
    else:
        nombre = input("Nombre de la biblioteca: ").strip() or "Mi Biblioteca"
        bib = Biblioteca(nombre)

    while True:
        print(MENU)
        opcion = input("Opción: ").strip()
        try:
            if opcion == "0":
                persistencia.guardar(bib, ARCHIVO)
                print("Guardado. ¡Hasta pronto!")
                break
            elif opcion == "1":
                for m in bib.listar_catalogo():
                    print(" ", m)
            elif opcion == "2":
                isbn  = input("ISBN: ").strip()
                titulo = input("Título: ").strip()
                autor  = input("Autor: ").strip()
                anio   = int(input("Año: ").strip())
                bib.agregar_material(Libro(isbn, titulo, anio, autor))
                print("Libro añadido.")
            elif opcion == "3":
                id_u  = input("ID usuario: ").strip()
                nombre = input("Nombre: ").strip()
                email  = input("Email: ").strip()
                bib.registrar_usuario(Usuario(id_u, nombre, email))
                print("Usuario registrado.")
            elif opcion == "4":
                id_u = input("ID usuario: ").strip()
                isbn = input("ISBN: ").strip()
                p = bib.prestar(id_u, isbn)
                print(f"Préstamo {p.id_prestamo} creado.")
            elif opcion == "5":
                id_p = input("ID préstamo: ").strip()
                bib.devolver(id_p)
                print("Devolución registrada.")
            elif opcion == "6":
                for p in bib.prestamos_activos():
                    print(f"  {p.id_prestamo}: usuario={p.id_usuario} isbn={p.isbn_material} desde={p.fecha_prestamo}")
            elif opcion == "7":
                r = bib.reporte_estado()
                for k, v in r.items():
                    print(f"  {k}: {v}")
        except BibliotecaError as e:
            print(f"  ⚠️  {e}")
        except ValueError as e:
            print(f"  ⚠️  Valor incorrecto: {e}")

if __name__ == "__main__":
    main()
""", encoding="utf-8")

!git add cli.py
!git commit -m "feat: CLI interactivo con menú completo"

!git switch main
!git merge feature/cli --no-ff -m "Merge: interfaz de línea de comandos"
!git branch -d feature/cli

# Etiquetar la primera versión estable
!git tag -a v1.0.0 -m "v1.0.0 — Primera versión estable del sistema"

print("\n✅ v1.0.0 etiquetada")
print("\nHistorial completo:")
!git log --oneline --graph

---
## Fase 4 — Añadir tests y etiquetar v1.1.0

In [ ]:
!git switch -c feature/tests

Path("tests/test_biblioteca.py").write_text("""\
import sys
sys.path.insert(0, str(__import__('pathlib').Path(__file__).parent.parent))

from biblioteca import Biblioteca
from material import Libro
from usuario import Usuario
from excepciones import MaterialNoDisponible, LimitePrestamosAlcanzado

def crear_bib_demo():
    bib = Biblioteca("Test")
    bib.agregar_material(Libro("ISBN-001", "Python Básico", 2023, "Autor A"))
    bib.agregar_material(Libro("ISBN-002", "Git Avanzado", 2022, "Autor B"))
    bib.registrar_usuario(Usuario("U1", "Ana", "ana@test.com"))
    return bib

def test_prestar_ok():
    bib = crear_bib_demo()
    p = bib.prestar("U1", "ISBN-001")
    assert p.activo
    assert not bib.buscar_material("ISBN-001").disponible
    print("✅ test_prestar_ok")

def test_prestar_no_disponible():
    bib = crear_bib_demo()
    bib.prestar("U1", "ISBN-001")
    try:
        bib.prestar("U1", "ISBN-001")
        assert False, "Debería lanzar MaterialNoDisponible"
    except MaterialNoDisponible:
        print("✅ test_prestar_no_disponible")

def test_devolver_ok():
    bib = crear_bib_demo()
    p = bib.prestar("U1", "ISBN-001")
    bib.devolver(p.id_prestamo)
    assert bib.buscar_material("ISBN-001").disponible
    print("✅ test_devolver_ok")

def test_limite_prestamos():
    bib = Biblioteca("Test")
    for i in range(4):
        bib.agregar_material(Libro(f"ISBN-{i:03d}", f"Libro {i}", 2023, "Autor"))
    bib.registrar_usuario(Usuario("U1", "Ana", "ana@test.com"))
    for i in range(3):  # tomar hasta el límite
        bib.prestar("U1", f"ISBN-{i:03d}")
    try:
        bib.prestar("U1", "ISBN-003")  # superar el límite
        assert False, "Debería lanzar LimitePrestamosAlcanzado"
    except LimitePrestamosAlcanzado:
        print("✅ test_limite_prestamos")

if __name__ == "__main__":
    test_prestar_ok()
    test_prestar_no_disponible()
    test_devolver_ok()
    test_limite_prestamos()
    print("\n✅ Todos los tests pasaron")
""", encoding="utf-8")

!git add tests/test_biblioteca.py
!git commit -m "test: suite básica de tests para Biblioteca"

# Ejecutar los tests para verificar
print("\nEjecutando tests:")
!python tests/test_biblioteca.py

!git switch main
!git merge feature/tests --no-ff -m "Merge: suite de tests"
!git branch -d feature/tests

!git tag -a v1.1.0 -m "v1.1.0 — Añadir suite de tests"
print("\n✅ v1.1.0 etiquetada")

---
## Práctica — Auditoría y extensión del proyecto

Completadas las fases guiadas, ahora trabaja de forma autónoma en los ejercicios.

---

### Ejercicio 1 — Auditoría del historial

**Enunciado:** Escribe una función `auditar_historial(ruta)` que analice el historial del proyecto y genere un informe con:

1. Número total de commits.
2. Distribución por tipo (feat, fix, docs, test, chore, Merge) con recuento y porcentaje.
3. Commits que NO siguen el formato Conventional Commits (no tienen prefijo `tipo:`).
4. Lista de tags con la fecha de cada versión.
5. Número de archivos únicos modificados en toda la historia.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
import subprocess, os
from pathlib import Path

os.chdir(Path.home() / "sistema_biblioteca")
print(f"Trabajando en: {Path.cwd()}")


<details>
<summary>💡 Ver solución — Ejercicio 1</summary>

```python
import subprocess, re
from pathlib import Path
from collections import Counter

def auditar_historial(ruta):
    ruta = Path(ruta).expanduser().resolve()

    def git(args):
        r = subprocess.run(["git"] + args, cwd=str(ruta),
                           capture_output=True, text=True)
        return r.stdout.strip()

    # 1. Commits totales
    logs = git(["log", "--pretty=format:%H|%s"]).splitlines()
    total = len(logs)

    # 2. Distribución por tipo
    prefijos_validos = ("feat", "fix", "docs", "test", "chore", "refactor", "style")
    tipos = Counter()
    sin_prefijo = []
    for linea in logs:
        _, msg = linea.split("|", 1)
        if msg.startswith("Merge"):
            tipos["Merge"] += 1
            continue
        match = re.match(r'^(\w+)[:(]', msg)
        if match and match.group(1) in prefijos_validos:
            tipos[match.group(1)] += 1
        else:
            tipos["otros"] += 1
            sin_prefijo.append(msg)

    # 4. Tags
    tags_raw = git(["tag", "-l", "--sort=version:refname"]).splitlines()
    tags_info = []
    for tag in tags_raw:
        fecha = git(["log", "-1", "--format=%ad", "--date=short", tag])
        tags_info.append((tag, fecha))

    # 5. Archivos únicos modificados
    archivos_raw = git(["log", "--name-only", "--pretty=format:"]).splitlines()
    archivos = {a for a in archivos_raw if a.strip()}

    # Imprimir informe
    print(f"\n{'═'*55}")
    print(f"  AUDITORÍA DE HISTORIAL — {ruta.name}")
    print(f"{'═'*55}")
    print(f"  Total de commits: {total}")

    print(f"\n  DISTRIBUCIÓN POR TIPO:")
    for tipo, count in tipos.most_common():
        pct = count / total * 100
        barra = "█" * int(pct / 5)
        print(f"    {tipo:<12} {count:>3}  {pct:5.1f}%  {barra}")

    print(f"\n  COMMITS SIN PREFIJO CONVENCIONAL ({len(sin_prefijo)}):")
    for m in sin_prefijo:
        print(f"    - {m[:60]}")
    if not sin_prefijo:
        print("    ✅ Todos los commits siguen Conventional Commits")

    print(f"\n  VERSIONES (tags):")
    for tag, fecha in tags_info:
        print(f"    {tag:<12}  {fecha}")

    print(f"\n  Archivos únicos modificados: {len(archivos)}")
    for a in sorted(archivos):
        print(f"    {a}")

    print(f"{'═'*55}")


auditar_historial("~/sistema_biblioteca")
```
</details>

---
### Ejercicio 2 — Hotfix sobre v1.0.0

**Enunciado:** Descubres que en `v1.0.0` hay un bug: `buscar_por_titulo` es sensible a mayúsculas. Aplica el flujo de hotfix:

1. Crea la rama `hotfix/busqueda-case-insensitive` desde `v1.0.0` (no desde `main`).
2. Corrige el bug en `biblioteca.py`: la búsqueda ya no distingue mayúsculas/minúsculas.
3. Commitea con mensaje apropiado.
4. Fusiona el hotfix en `main`.
5. Etiqueta `v1.0.1`.
6. Muestra el historial gráfico.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
# Pista: git switch -c hotfix/... v1.0.0


<details>
<summary>💡 Ver solución — Ejercicio 2</summary>

```python
import os, subprocess
from pathlib import Path

os.chdir(Path.home() / "sistema_biblioteca")

# 1. Crear hotfix desde el tag v1.0.0
!git switch -c hotfix/busqueda-case-insensitive v1.0.0

# 2. Corregir el bug
# buscar_por_titulo ya usaba .lower() → en realidad está bien.
# Vamos a añadir también búsqueda en el tipo y descripción_corta.
contenido = Path("biblioteca.py").read_text(encoding="utf-8")
contenido = contenido.replace(
    'return [m for m in self._catalogo if t in m.titulo.lower()]',
    'return [m for m in self._catalogo if t in m.titulo.lower() or t in m.tipo.lower()]'
)
Path("biblioteca.py").write_text(contenido, encoding="utf-8")

# 3. Commit
!git add biblioteca.py
!git commit -m "fix: buscar_por_titulo incluye búsqueda en tipo de material"

# 4. Fusionar en main
!git switch main
!git merge hotfix/busqueda-case-insensitive --no-ff -m "Merge: hotfix búsqueda case-insensitive"
!git branch -d hotfix/busqueda-case-insensitive

# 5. Etiquetar
!git tag -a v1.0.1 -m "v1.0.1 — Hotfix: busqueda por tipo de material"

# 6. Historial
!git log --oneline --graph
print("\nTags:")
!git tag -l
```

**Nota:** En Git flow real, el hotfix también se fusionaría en `develop` (si existiera). En proyectos pequeños con solo `main`, basta con fusionar en `main`.
</details>

---
### Ejercicio 3 — Nueva feature: estadísticas

**Enunciado:** Añade al sistema una funcionalidad de estadísticas siguiendo el flujo de feature branch:

1. Crea la rama `feature/estadisticas` desde `main`.
2. Añade el método `estadisticas_avanzadas()` en `Biblioteca` que devuelva:
   - Material más prestado (isbn, titulo, veces_prestado)
   - Usuario con más préstamos históricos
   - Promedio de días por préstamo devuelto
3. Commitea.
4. Fusiona en `main` con `--no-ff`.
5. Etiqueta `v1.2.0`.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


---
### Ejercicio 4 — Publicar en GitHub con estructura completa

**Enunciado:** Publica el proyecto `sistema_biblioteca` en GitHub:

1. Crea el repositorio en GitHub (sin inicializar).
2. Añade el remote `origin`.
3. Sube `main` con todos sus commits.
4. Sube todos los tags con `git push --tags`.
5. Comprueba en GitHub que:
   - Aparecen todas las ramas (solo `main` — las feature branches se borraron)
   - Aparecen los tags `v1.0.0`, `v1.0.1`, `v1.1.0`, `v1.2.0`
   - El historial muestra el grafo de merges
6. Escribe la URL del repositorio publicado.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
# !git remote add origin git@github.com:TU-USUARIO/sistema-biblioteca.git
# !git push -u origin main
# !git push --tags


---
### Ejercicio 5 (reto) — Generador de CHANGELOG

**Enunciado:** Escribe la función `generar_changelog(ruta)` que lea el historial Git y genere automáticamente un archivo `CHANGELOG.md` con formato profesional:

```markdown
# Changelog

## [v1.2.0] — 2024-01-15

### Nuevas funcionalidades
- feat: estadísticas avanzadas de préstamos

### Correcciones
- (ninguna)

## [v1.1.0] — 2024-01-10

### Nuevas funcionalidades
- feat: ...
...
```

La función debe:
- Obtener los tags ordenados cronológicamente.
- Para cada par de tags consecutivos, listar los commits entre ellos.
- Clasificarlos en `feat`, `fix`, `docs`, `test`, `otros`.
- Escribir `CHANGELOG.md` y hacer commit con `docs: generar CHANGELOG.md`.

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
import subprocess
from pathlib import Path


<details>
<summary>💡 Ver solución — Ejercicio 5</summary>

```python
import subprocess, re, os
from pathlib import Path

def generar_changelog(ruta):
    ruta = Path(ruta).expanduser().resolve()
    os.chdir(ruta)

    def git(args):
        r = subprocess.run(["git"] + args, cwd=str(ruta),
                           capture_output=True, text=True)
        return r.stdout.strip()

    # Tags ordenados cronológicamente
    tags_raw = git(["tag", "-l", "--sort=version:refname"]).splitlines()
    if not tags_raw:
        print("No hay tags — no se puede generar changelog")
        return

    def commits_entre(desde, hasta):
        rango = f"{desde}..{hasta}" if desde else hasta
        raw = git(["log", rango, "--pretty=format:%s", "--no-merges"])
        return [l for l in raw.splitlines() if l.strip()]

    def clasificar(commits):
        grupos = {"feat": [], "fix": [], "docs": [], "test": [], "otros": []}
        for c in commits:
            match = re.match(r'^(feat|fix|docs|test|chore|refactor|style)[:(]', c)
            clave = match.group(1) if match else "otros"
            if clave not in grupos:
                clave = "otros"
            grupos[clave].append(c)
        return grupos

    def fecha_tag(tag):
        return git(["log", "-1", "--format=%ad", "--date=short", tag])

    secciones = {
        "feat": "Nuevas funcionalidades",
        "fix": "Correcciones",
        "docs": "Documentación",
        "test": "Tests",
        "otros": "Otros cambios",
    }

    lineas = ["# Changelog\n", "",
              "Generado automáticamente desde el historial Git.\n"]

    pares = [(tags_raw[i-1] if i > 0 else None, tags_raw[i])
             for i in range(len(tags_raw)-1, -1, -1)]  # de más nuevo a más antiguo

    for desde, hasta in pares:
        commits = commits_entre(desde, hasta)
        fecha = fecha_tag(hasta)
        lineas.append(f"## [{hasta}] — {fecha}\n")

        grupos = clasificar(commits)
        algo = False
        for clave, titulo in secciones.items():
            if grupos[clave]:
                lineas.append(f"### {titulo}")
                for c in grupos[clave]:
                    lineas.append(f"- {c}")
                lineas.append("")
                algo = True
        if not algo:
            lineas.append("_(sin cambios registrados)_\n")

    changelog = "\n".join(lineas)
    Path("CHANGELOG.md").write_text(changelog, encoding="utf-8")
    print(changelog)

    subprocess.run(["git", "add", "CHANGELOG.md"], cwd=str(ruta))
    subprocess.run(["git", "commit", "-m", "docs: generar CHANGELOG.md automáticamente"],
                   cwd=str(ruta))
    print("\n✅ CHANGELOG.md generado y commiteado")


generar_changelog("~/sistema_biblioteca")
```
</details>

---
## Reflexión final — ¿Qué hemos construido?

A lo largo de este proyecto has aplicado, sin que te lo indiquen paso a paso:

| Habilidad | Dónde la usaste |
|-----------|----------------|
| `git init` + `.gitignore` | Fase 1 |
| Commits convencionales | Todas las fases |
| Feature branches + merge --no-ff | Fases 2, 3 y 4 |
| `git tag -a` | v1.0.0, v1.1.0, v1.0.1, v1.2.0 |
| Hotfix desde un tag | Ejercicio 2 |
| `git push` + `git push --tags` | Ejercicio 4 |
| Auditoría con `subprocess` | Ejercicio 1 y 5 |
| Generación de CHANGELOG | Ejercicio 5 |

**Lo más importante no es que el código funcione — es que el historial cuenta una historia.** Alguien que no ha visto el proyecto puede leer `git log --oneline --graph` y entender qué se construyó, en qué orden y por qué.

> El código es para las máquinas. El historial Git es para las personas.

---
## Módulo 2 completado — Resumen de lo aprendido

```
CLASE 1  Git Introducción
         ├── Problema que resuelve Git
         ├── Tres zonas: working / staging / repo
         ├── init, add, commit, log
         └── .gitignore

CLASE 2  Git Local Avanzado
         ├── diff, restore, revert, reset
         ├── stash, amend
         ├── log avanzado, blame
         └── tags

CLASE 3  GitHub y Remotos
         ├── SSH vs HTTPS
         ├── remote add, push, clone
         ├── fetch vs pull
         └── Fork y upstream

CLASE 4  Ramas y Colaboración
         ├── branch, switch
         ├── merge: fast-forward y 3-way
         ├── Resolución de conflictos
         ├── Git flow simplificado
         └── Pull Requests

CLASE 5  Proyecto: Sistema Biblioteca
         ├── Historial real con feature branches
         ├── Hotfix desde un tag
         ├── Publicación en GitHub con tags
         └── Auditoría y CHANGELOG automáticos
```

**Próximo módulo:** Python para datos — NumPy, Pandas y visualización.